# Week 4 – Simple ETL Pipeline in Azure Databricks

# Step 1: PySpark and Delta setup

In [2]:
# Install PySpark
!pip install pyspark -q

In [3]:
# Import the Requirements
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, current_date, datediff


In [4]:
#Install dependencies (only needed in Google Colab)
# In Azure Databricks, skip this cell
!pip install pyspark==3.5.1 delta-spark==3.1.0
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip



In [5]:
# using DELTA in colab
!pip install delta-spark==3.2.0 -q
import pyspark
from delta import *
from pyspark.sql.functions import *

# Create a SparkSession with Delta Lake extensions
# The '.config(...)' lines are crucial for enabling Delta Lake's features
builder = pyspark.sql.SparkSession.builder.appName("DeltaDemo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

# Get or create the SparkSession
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark and Delta Lake are ready!")

Spark and Delta Lake are ready!


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Step 2: Upload the CSV files

In [7]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders.csv
Saving delivery_status.csv to delivery_status.csv
Saving customers.csv to customers.csv


# Step 3:  Load cleaned order data into Databricks

In [8]:
from google.colab import files
uploaded = files.upload()

Saving cleaned_orders.csv to cleaned_orders.csv


# Step 4: Load CSV into Spark DataFrame

In [9]:
csv_filename = list(uploaded.keys())[0]

# Load into Spark DataFrame
cleaned_orders = spark.read.csv(csv_filename, header=True, inferSchema=True)

print("Cleaned Orders Data:")
cleaned_orders.show()

Cleaned Orders Data:
+--------+-----------+----------+-------------+---------+
|order_id|customer_id|order_date|delivery_date|   status|
+--------+-----------+----------+-------------+---------+
|       1|          1|2025-07-01|   2025-07-03|Delivered|
|       2|          2|2025-07-05|   2025-07-08|Delivered|
|       3|          3|2025-07-10|   2025-07-12|  Pending|
|       4|          4|2025-07-12|   2025-07-15|  Shipped|
|       5|          1|2025-07-18|   2025-07-20|  Pending|
|       6|          5|2025-07-20|   2025-07-23|Delivered|
|       7|          6|2025-07-22|   2025-07-25|  Shipped|
+--------+-----------+----------+-------------+---------+



# Step 5: Create a pipeline to update latest delivery status

In [12]:
# Upload Orders file
print("Upload CLeaned orders.csv")
uploaded_orders = files.upload()
orders_file = list(uploaded_orders.keys())[0]
orders_df = spark.read.csv(orders_file, header=True, inferSchema=True)

# Upload Delivery Status file
print("Upload delivery_status.csv")
uploaded_status = files.upload()
status_file = list(uploaded_status.keys())[0]
delivery_status_df = spark.read.csv(status_file, header=True, inferSchema=True)



Upload CLeaned orders.csv


Saving cleaned_orders.csv to cleaned_orders (3).csv
Upload delivery_status.csv


Saving delivery_status.csv to delivery_status (3).csv


In [13]:
# Rename last_updated
delivery_status_df = delivery_status_df.withColumnRenamed("last_updated", "status_update_time")

# Join orders with latest delivery status
orders_updated_df = (
    orders_df.alias("orders")
    .join(delivery_status_df.alias("status"), on="order_id", how="left")
    .select(
        "orders.*",
        F.col("status.current_status").alias("latest_status"),
        "status.status_update_time"
    )
)

# Calculate delay_days
orders_updated_df = orders_updated_df.withColumn(
    "delay_days",
    F.when(
        (F.col("delivery_date").isNotNull()) & (F.col("status_update_time").isNotNull()),
        F.datediff(F.to_date(F.col("status_update_time")), F.to_date(F.col("delivery_date")))
    ).otherwise(None)
)

# Add delayed flag (1 if delayed, else 0)
orders_updated_df = orders_updated_df.withColumn(
    "delayed",
    F.when(F.col("delay_days") > 0, 1).otherwise(0)
)

# Show final updated orders data with latest delivery status
print("Updated Orders Data with latest delivery status:")
orders_updated_df.show(truncate=False)

Updated Orders Data with latest delivery status:
+--------+-----------+----------+-------------+---------+-------------+-------------------+----------+-------+
|order_id|customer_id|order_date|delivery_date|status   |latest_status|status_update_time |delay_days|delayed|
+--------+-----------+----------+-------------+---------+-------------+-------------------+----------+-------+
|1       |1          |2025-07-01|2025-07-03   |Delivered|Delivered    |2025-07-03 10:00:00|0         |0      |
|2       |2          |2025-07-05|2025-07-08   |Delivered|Delivered    |2025-07-08 15:30:00|0         |0      |
|3       |3          |2025-07-10|2025-07-12   |Pending  |In Transit   |2025-07-20 11:45:00|8         |1      |
|4       |4          |2025-07-12|2025-07-15   |Shipped  |Shipped      |2025-07-21 09:00:00|6         |1      |
|5       |1          |2025-07-18|2025-07-20   |Pending  |Pending      |2025-07-23 08:00:00|3         |1      |
|6       |5          |2025-07-20|2025-07-23   |Delivered|Delive

# Step 6: Save the results as Delta or CSV

In [14]:
# Save as Delta
orders_updated_df.write.format("delta").mode("overwrite").save("/content/orders_with_status_delta")

# Save as CSV
orders_updated_df.write.mode("overwrite").option("header", "true").csv("/content/orders_with_status_csv")


# Step 7: SQL query to show top 5 delayed customers

In [15]:
# Register DataFrame as a temp SQL view
orders_updated_df.createOrReplaceTempView("orders_with_status")

top_delayed_customers = spark.sql("""
    SELECT customer_id,
           COUNT(*) AS total_delays
    FROM orders_with_status
    WHERE delay_days > 0
    GROUP BY customer_id
    ORDER BY total_delays DESC
    LIMIT 5
""")

print("Top 5 Delayed Customers:")
top_delayed_customers.show()


Top 5 Delayed Customers:
+-----------+------------+
|customer_id|total_delays|
+-----------+------------+
|          1|           1|
|          3|           1|
|          4|           1|
+-----------+------------+



#Deliverables:

# Output stored in Delta/CSV

In [16]:
# Save as Single CSV file
single_csv_path = "/content/orders_with_status_single_csv"
orders_updated_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(single_csv_path)

# Find the part file
import glob, shutil
part_file = glob.glob(f"{single_csv_path}/*.csv")[0]

# Rename to clean filename
final_csv_path = "/content/orders_with_delivery_status.csv"
shutil.move(part_file, final_csv_path)

# Download in Colab
from google.colab import files
files.download(final_csv_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>